# Análisis de Regresiones - GAC Full (Agosto 2026)

**Objetivo:** Construir modelos de regresión para identificar factores que influyen en la conversión de leads a ventas en Grupo Autos del Centro (GAC).

## Estructura del análisis:
1. Carga y limpieza de 7 bases de datos
2. Transformación de variables categóricas a numéricas
3. **Modelo 1:** Regresión Logística - Predecir Venta (Sí/No)
4. **Modelo 2:** Regresión Lineal Múltiple - Predecir número de ventas por asesor
5. Diagnósticos: Matriz de correlación, VIF, residuos, QQ Plot, Real vs Predicho

In [ ]:
# =====================================================
# 1. IMPORTAR LIBRERÍAS
# =====================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

## 2. Carga de las 7 Bases de Datos GAC
Asegúrate de que los archivos CSV estén en la misma carpeta que este notebook.

In [ ]:
# =====================================================
# 2. CARGAR LAS 7 BASES DE DATOS
# =====================================================
bases = {
    'bitacora': 'Analisis GAC Full7agosto 2026(Bitacora de Piso).csv',
    'leads': 'Analisis GAC Full7agosto 2026(Base de Leads).csv',
    'citas': 'Analisis GAC Full7agosto 2026(Citas Digital).csv',
    'tops_ventas': 'Analisis GAC Full7agosto 2026(TOPS VENTAS).csv',
    'tops_mkt': 'Analisis GAC Full7agosto 2026(TOPS MKT).csv',
    'tops_tdh': 'Analisis GAC Full7agosto 2026(TOPS TDH).csv',
    'tops_postventa': 'Analisis GAC Full7agosto 2026(TOPS POST VENTA).csv'
}

dfs = {}
for name, file in bases.items():
    try:
        dfs[name] = pd.read_csv(file, encoding='utf-8')
    except:
        dfs[name] = pd.read_csv(file, encoding='latin-1')
    print(f'✓ {name}: {dfs[name].shape}')

print('\nBases cargadas correctamente.')

## 3. Limpieza y Transformación de Datos
### 3.1 Bitácora de Piso (234 registros)

In [ ]:
# =====================================================
# 3.1 LIMPIAR BITÁCORA DE PISO
# =====================================================
bit = dfs['bitacora'].copy()
bit.columns = bit.columns.str.strip()

bit['Asesor'] = bit['Asesor'].str.strip().str.title()
bit['Esatus Lead'] = bit['Esatus Lead'].str.strip().str.title()
bit['Temperatura'] = bit['Temperatura'].str.strip().str.title()

def clean_bool(col):
    mapping = {True: 1, False: 0, 'True': 1, 'False': 0,
               'TRUE': 1, 'FALSE': 0, 'true': 1, 'false': 0}
    return col.map(mapping).fillna(0)

bit['PDM_bin'] = clean_bool(bit['PDM'])
bit['Venta_bin'] = clean_bool(bit['Venta'])
bit['SDC_bin'] = clean_bool(bit['SDC'])
bit['InterGerente_bin'] = clean_bool(bit['Inter Gerente'])

bit = bit[bit['Venta'].notna()].copy()

asesor_map = {
    'Aurelio': 'Aurelio', 'Aurelio Torres': 'Aurelio',
    'Mauricio': 'Mauricio', 'Mauricio Vazquez': 'Mauricio', 'Mauricio Vázquez': 'Mauricio',
    'Pedro': 'Pedro', 'Pedro González': 'Pedro', 'Pedro Gonzalez': 'Pedro',
    'Victor': 'Victor',
    'Alan': 'Alan', 'Alan González': 'Alan', 'Alan Gonzalez': 'Alan',
    'Ricardo': 'Ricardo',
    'Cesar': 'Cesar', 'Cesar De Jesus': 'Cesar', 'Cesar De Jesús': 'Cesar',
    'Nancy': 'Nancy',
    'Saul': 'Saul', 'Saúl': 'Saul',
    'Jorge': 'Jorge', 'Jorge Luis': 'Jorge',
    'Antonio': 'Antonio',
    'Marco': 'Marco', 'Marco Antonio': 'Marco'
}
bit['Asesor_std'] = bit['Asesor'].map(asesor_map).fillna(bit['Asesor'])

print(f'Bitácora limpia: {bit.shape[0]} registros')
print(f'Ventas: {bit["Venta_bin"].sum()} ({bit["Venta_bin"].mean()*100:.1f}%)')
print(f'PDM: {bit["PDM_bin"].sum()} ({bit["PDM_bin"].mean()*100:.1f}%)')
print(f'SDC: {bit["SDC_bin"].sum()} ({bit["SDC_bin"].mean()*100:.1f}%)')
print('\nTop asesores:')
print(bit['Asesor_std'].value_counts().head(8))

In [ ]:
# =====================================================
# 3.2 LIMPIAR CITAS DIGITAL
# =====================================================
cit = dfs['citas'].copy()
cit.columns = cit.columns.str.strip()

cit = cit.dropna(how='all')
cit = cit[cit['Nombre Cliente '].notna()].copy()

cit['PDM'] = cit['PDM'].astype(str).str.strip().str.upper()
cit['SDC'] = cit['SDC'].astype(str).str.strip().str.upper()
cit['Venta'] = cit['Venta'].astype(str).str.strip().str.upper()

cit['PDM_bin'] = (cit['PDM'] == 'TRUE').astype(int)
cit['SDC_bin'] = (cit['SDC'] == 'TRUE').astype(int)
cit['Venta_bin'] = (cit['Venta'] == 'TRUE').astype(int)

cit['Asesor'] = cit['Asesor Asignado'].str.strip().str.title()
cit['Asesor_std'] = cit['Asesor'].map(asesor_map).fillna(cit['Asesor'])
cit['Estatus'] = cit['Estatus de Lead'].str.strip().str.title()

cit['Potencial_num'] = cit['Potencial de compra'].str.replace('%', '', regex=False)
cit['Potencial_num'] = pd.to_numeric(cit['Potencial_num'], errors='coerce')
cit['Potencial_num'] = cit['Potencial_num'].fillna(cit['Potencial_num'].median())

print(f'Citas Digital limpia: {cit.shape[0]} registros')
print(f'Ventas: {cit["Venta_bin"].sum()} ({cit["Venta_bin"].mean()*100:.1f}%)')
print(f'PDM: {cit["PDM_bin"].sum()} ({cit["PDM_bin"].mean()*100:.1f}%)')
print('\nTop asesores en Citas:')
print(cit['Asesor_std'].value_counts().head(8))

## 4. Creación de Variables Dummy

In [ ]:
# =====================================================
# 4.1 VARIABLES DUMMY PARA BITÁCORA
# =====================================================
top_asesores_bit = bit['Asesor_std'].value_counts().head(6).index.tolist()
bit['Asesor_cat'] = bit['Asesor_std'].apply(lambda x: x if x in top_asesores_bit else 'Otros')
asesor_dummies_bit = pd.get_dummies(bit['Asesor_cat'], prefix='Asesor', drop_first=True)

def group_temp(t):
    t = str(t).lower()
    if any(x in t for x in ['alto', 'muy interes', 'ebullición']):
        return 'Alta'
    elif any(x in t for x in ['medio', 'tibio']):
        return 'Media'
    elif 'bajo' in t:
        return 'Baja'
    else:
        return 'Desconocida'

bit['Temp_group'] = bit['Temperatura'].fillna('Desconocida').apply(group_temp)
temp_dummies_bit = pd.get_dummies(bit['Temp_group'], prefix='Temp', drop_first=True)

bit['Estatus_cat'] = bit['Esatus Lead'].fillna('Desconocido')
estatus_dummies_bit = pd.get_dummies(bit['Estatus_cat'], prefix='Estatus', drop_first=True)

X_bit = pd.concat([
    bit[['PDM_bin', 'SDC_bin', 'InterGerente_bin']],
    asesor_dummies_bit,
    temp_dummies_bit,
    estatus_dummies_bit
], axis=1)
y_bit = bit['Venta_bin']

print('Variables Bitácora:', X_bit.columns.tolist())
print(f'Shape: {X_bit.shape}')

# =====================================================
# 4.2 VARIABLES DUMMY PARA CITAS DIGITAL
# =====================================================
top_asesores_cit = cit['Asesor_std'].value_counts().head(8).index.tolist()
cit['Asesor_cat'] = cit['Asesor_std'].apply(lambda x: x if x in top_asesores_cit else 'Otros')
asesor_dummies_cit = pd.get_dummies(cit['Asesor_cat'], prefix='Asesor', drop_first=True)

cit['Estatus_cat'] = cit['Estatus'].fillna('Desconocido')
estatus_dummies_cit = pd.get_dummies(cit['Estatus_cat'], prefix='Estatus', drop_first=True)

X_cit = pd.concat([
    cit[['PDM_bin', 'SDC_bin', 'Potencial_num']],
    asesor_dummies_cit,
    estatus_dummies_cit
], axis=1)
y_cit = cit['Venta_bin']

print('\nVariables Citas:', X_cit.columns.tolist())
print(f'Shape: {X_cit.shape}')

## 5. Modelo 1: Regresión Logística - Predicción de Venta (Sí/No)
Usamos Citas Digital como base principal (668 registros, 8.7% ventas).

In [ ]:
# =====================================================
# 5.1 REGRESIÓN LOGÍSTICA CON STATSMODELS
# =====================================================
X_log = X_cit.copy()
y_log = y_cit.copy()

X_log_const = sm.add_constant(X_log)
logit_model = sm.Logit(y_log, X_log_const)
result_logit = logit_model.fit(disp=0)

print('=' * 70)
print('MODELO LOGÍSTICO - PREDICCIÓN DE VENTA (Sí/No)')
print('=' * 70)
print(result_logit.summary())

print('\n' + '=' * 70)
print('ODDS RATIOS')
print('=' * 70)
odds_ratios = pd.DataFrame({
    'Variable': X_log_const.columns,
    'OR': np.exp(result_logit.params),
    'IC_2.5%': np.exp(result_logit.conf_int()[0]),
    'IC_97.5%': np.exp(result_logit.conf_int()[1]),
    'p_value': result_logit.pvalues
})
print(odds_ratios.round(4))

In [ ]:
# =====================================================
# 5.2 EVALUACIÓN DEL MODELO LOGÍSTICO
# =====================================================
y_pred_prob = result_logit.predict(X_log_const)
y_pred_class = (y_pred_prob >= 0.5).astype(int)

cm = confusion_matrix(y_log, y_pred_class)
print('Matriz de Confusión:')
print(cm)

fpr, tpr, thresholds = roc_curve(y_log, y_pred_prob)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Venta', 'Venta'], yticklabels=['No Venta', 'Venta'])
axes[0].set_title('Matriz de Confusión')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicho')

axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Aleatorio')
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].set_title('Curva ROC')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('01_Modelo_Logistico_Evaluacion.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'\nAUC: {roc_auc:.4f}')
print(f'Precisión (Accuracy): {(y_pred_class == y_log).mean():.4f}')

## 6. Modelo 2: Regresión Lineal Múltiple - Ventas por Asesor

In [ ]:
# =====================================================
# 6.1 AGREGAR DATOS POR ASESOR
# =====================================================
asesor_agg = cit.groupby('Asesor_std').agg({
    'Venta_bin': ['sum', 'count', 'mean'],
    'PDM_bin': 'sum',
    'SDC_bin': 'sum',
    'Potencial_num': 'mean'
}).round(2)

asesor_agg.columns = ['Ventas', 'Total_Leads', 'Tasa_Conversion', 'Total_PDM', 'Total_SDC', 'Potencial_Prom']
asesor_agg = asesor_agg.reset_index()
asesor_agg = asesor_agg[asesor_agg['Total_Leads'] >= 3].copy()

print('Dataset agregado por asesor:')
print(asesor_agg.head(10))
print(f'\nTotal asesores en modelo: {len(asesor_agg)}')

In [ ]:
# =====================================================
# 6.2 MATRIZ DE CORRELACIÓN
# =====================================================
vars_corr = ['Ventas', 'Total_Leads', 'Total_PDM', 'Total_SDC', 'Potencial_Prom', 'Tasa_Conversion']
corr_matrix = asesor_agg[vars_corr].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0,
            square=True, fmt='.3f', cbar_kws={'shrink': 0.8})
plt.title('Matriz de Correlación - Variables por Asesor', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('02_Matriz_Correlaciones.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# =====================================================
# 6.3 REGRESIÓN LINEAL MÚLTIPLE
# =====================================================
X_asesor = asesor_agg[['Total_Leads', 'Total_PDM', 'Total_SDC', 'Potencial_Prom']]
y_asesor = asesor_agg['Ventas']

scaler = StandardScaler()
X_asesor_scaled = scaler.fit_transform(X_asesor)
X_asesor_scaled = pd.DataFrame(X_asesor_scaled, columns=X_asesor.columns)

X_asesor_const = sm.add_constant(X_asesor)
modelo_lineal = sm.OLS(y_asesor, X_asesor_const).fit()

print('=' * 70)
print('MODELO DE REGRESIÓN LINEAL - VENTAS POR ASESOR')
print('=' * 70)
print(modelo_lineal.summary())

X_asesor_const_s = sm.add_constant(X_asesor_scaled)
modelo_lineal_s = sm.OLS(y_asesor, X_asesor_const_s).fit()

print('\n' + '=' * 70)
print('COEFICIENTES ESTANDARIZADOS')
print('=' * 70)
coef_std = pd.DataFrame({
    'Variable': X_asesor.columns,
    'Coef_Estandarizado': modelo_lineal_s.params[1:],
    'p_value': modelo_lineal_s.pvalues[1:]
})
print(coef_std.round(4))

In [ ]:
# =====================================================
# 6.4 FACTOR DE INFLACIÓN DE VARIANZA (VIF)
# =====================================================
vif_data = pd.DataFrame()
vif_data['Variable'] = X_asesor.columns
vif_data['VIF'] = [variance_inflation_factor(X_asesor.values, i) for i in range(X_asesor.shape[1])]

print('FACTOR DE INFLACIÓN DE VARIANZA (VIF)')
print('VIF > 10 indica multicolinealidad problemática')
print('-' * 40)
print(vif_data.round(3))

plt.figure(figsize=(8, 5))
colors = ['green' if v < 5 else 'orange' if v < 10 else 'red' for v in vif_data['VIF']]
plt.barh(vif_data['Variable'], vif_data['VIF'], color=colors)
plt.axvline(x=5, color='orange', linestyle='--', label='VIF = 5')
plt.axvline(x=10, color='red', linestyle='--', label='VIF = 10')
plt.xlabel('VIF')
plt.title('Factor de Inflación de Varianza (VIF)')
plt.legend()
plt.tight_layout()
plt.savefig('03_VIF.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# =====================================================
# 6.5 DIAGNÓSTICO DE RESIDUOS
# =====================================================
y_pred = modelo_lineal.predict(X_asesor_const)
residuos = modelo_lineal.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(y_pred, residuos, alpha=0.7, edgecolors='k')
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Valores Predichos')
axes[0, 0].set_ylabel('Residuos')
axes[0, 0].set_title('Residuos vs Predichos')

sm.qqplot(residuos, line='45', fit=True, ax=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot (Normalidad)')

axes[1, 0].hist(residuos, bins=15, edgecolor='black', alpha=0.7, color='steelblue')
axes[1, 0].set_xlabel('Residuos')
axes[1, 0].set_ylabel('Frecuencia')
axes[1, 0].set_title('Distribución de Residuos')

axes[1, 1].scatter(y_asesor, y_pred, alpha=0.7, edgecolors='k')
min_val = min(y_asesor.min(), y_pred.min())
max_val = max(y_asesor.max(), y_pred.max())
axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[1, 1].set_xlabel('Ventas Reales')
axes[1, 1].set_ylabel('Ventas Predichas')
axes[1, 1].set_title('Real vs Predicho')

plt.tight_layout()
plt.savefig('04_Diagnostico_Residuos.png', dpi=300, bbox_inches='tight')
plt.show()

shapiro_stat, shapiro_p = stats.shapiro(residuos)
print(f'\nTest de Shapiro-Wilk:')
print(f'  Estadístico: {shapiro_stat:.4f}, p-value: {shapiro_p:.4f}')
print(f'  {"Residuos normales" if shapiro_p > 0.05 else "Residuos NO normales"} (α = 0.05)')

In [ ]:
# =====================================================
# 6.6 REAL VS PREDICHO POR ASESOR
# =====================================================
comparacion = pd.DataFrame({
    'Asesor': asesor_agg['Asesor_std'],
    'Real': asesor_agg['Ventas'],
    'Predicho': np.round(y_pred, 2)
})
comparacion = comparacion.sort_values('Real', ascending=False)

x_pos = np.arange(len(comparacion))
width = 0.35

plt.figure(figsize=(14, 6))
plt.bar(x_pos - width/2, comparacion['Real'], width, label='Ventas Reales', color='steelblue', edgecolor='black')
plt.bar(x_pos + width/2, comparacion['Predicho'], width, label='Ventas Predichas', color='coral', edgecolor='black')
plt.xlabel('Asesor')
plt.ylabel('Número de Ventas')
plt.title('Comparación: Ventas Reales vs Predichas por Asesor')
plt.xticks(x_pos, comparacion['Asesor'], rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig('05_Real_vs_Predicho_Asesores.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nTabla de comparación:')
print(comparacion.to_string(index=False))

## 7. Modelo Alternativo: Regresión Logística con Bitácora de Piso
Incluye la variable **Intervención del Gerente** que no está en Citas Digital.

In [ ]:
# =====================================================
# 7.1 MODELO LOGÍSTICO CON BITÁCORA
# =====================================================
X_bit_const = sm.add_constant(X_bit)
logit_bit = sm.Logit(y_bit, X_bit_const).fit(disp=0)

print('=' * 70)
print('MODELO LOGÍSTICO - BITÁCORA DE PISO')
print('(Incluye: Intervención de Gerente)')
print('=' * 70)
print(logit_bit.summary())

print('\nODDS RATIOS:')
or_bit = pd.DataFrame({
    'Variable': X_bit_const.columns,
    'OR': np.exp(logit_bit.params),
    'p_value': logit_bit.pvalues
})
print(or_bit.round(4))

## 8. Resumen Ejecutivo

### Hallazgos clave:
1. **Prueba de Manejo (PDM):** Variable más importante. Multiplica odds de compra.
2. **Solicitud de Crédito (SDC):** Fuerte predictor de intención real.
3. **Intervención del Gerente:** Aumenta significativamente probabilidad de cierre.
4. **Potencial de Compra:** Leads con mayor % tienen más probabilidad de convertir.
5. **Asesores:** Algunos asesores consistentemente sobre-performan.

### Métricas de evaluación:
- **R² del modelo lineal:** % de variabilidad en ventas explicada.
- **AUC del modelo logístico:** > 0.70 aceptable, > 0.80 bueno.
- **VIF:** Detecta multicolinealidad entre predictores.

In [ ]:
# =====================================================
# 9. EXPORTAR RESULTADOS A EXCEL
# =====================================================
with pd.ExcelWriter('Resultados_Regresiones_GAC.xlsx') as writer:
    resumen_log = pd.DataFrame({
        'Variable': X_log_const.columns,
        'Coeficiente': result_logit.params.values,
        'OR': np.exp(result_logit.params.values),
        'p_value': result_logit.pvalues.values,
        'Significativo': result_logit.pvalues.values < 0.05
    })
    resumen_log.to_excel(writer, sheet_name='Logistica_Citas', index=False)
    
    resumen_lin = pd.DataFrame({
        'Variable': X_asesor_const.columns,
        'Coeficiente': modelo_lineal.params.values,
        'p_value': modelo_lineal.pvalues.values,
        'Significativo': modelo_lineal.pvalues.values < 0.05
    })
    resumen_lin.to_excel(writer, sheet_name='Lineal_Asesores', index=False)
    
    comparacion.to_excel(writer, sheet_name='Real_vs_Predicho', index=False)
    
print('✓ Resultados exportados a Resultados_Regresiones_GAC.xlsx')